In [3]:
#NAME:CHARANPAL SINGH A/L MANJEET SINGH
#ID:IS01083868
#NAME:NUR IDRAKI HAFIY BIN MOHAMAD ZAIDI 
#ID:IS01084550


"""
Discussion on Strengths and Weaknesses of Selected Models:
For this sentiment classification task, two models were evaluated: a lexicon-based approach (VADER) and a machine-learning approach (Logistic Regression using TF-IDF). 
- Strengths: VADER is excellent because it requires no training data and is very fast to implement out-of-the-box. Logistic Regression with TF-IDF is highly effective because it learns the specific vocabulary weights (like food-related adjectives) from our exact dataset, generally leading to higher accuracy.
- Weaknesses: VADER struggles with sarcasm and context specific to the food domain, as it relies on a generalized dictionary. Logistic Regression's main weakness is that it requires labeled training data, takes longer to train, and the TF-IDF matrix can consume a lot of memory.
"""

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('vader_lexicon')
nltk.download('stopwords')

# 1. Data Preprocessing
print("Loading and preprocessing data...")

# FIX 1 APPLIED: Read only the first 20,000 rows to avoid the EOF error, then sample 10,000
df = pd.read_csv('Reviews.csv', nrows=20000).sample(n=10000, random_state=42)

# Drop missing text values
df.dropna(subset=['Text', 'Score'], inplace=True)

# Create a binary sentiment label based on the 1-5 Score
# Score > 3 is Positive (1), Score < 3 is Negative (0). Drop neutral (3) for simplicity.
df = df[df['Score'] != 3]
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

# Basic text cleaning function
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation/numbers
    text = text.lower() # Lowercase
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

df['Clean_Text'] = df['Text'].apply(clean_text)

# 2. Model 1: Lexicon-Based Approach (VADER)
print("\n--- Evaluating Lexicon-Based Model (VADER) ---")
sia = SentimentIntensityAnalyzer()

# Apply VADER to get compound scores
df['Vader_Score'] = df['Clean_Text'].apply(lambda x: sia.polarity_scores(x)['compound'])
# Convert compound score to binary prediction (>0 is positive)
df['Vader_Prediction'] = df['Vader_Score'].apply(lambda x: 1 if x > 0 else 0)

print(f"VADER Accuracy: {accuracy_score(df['Sentiment'], df['Vader_Prediction']):.4f}")
print(classification_report(df['Sentiment'], df['Vader_Prediction']))


# 3. Feature Extraction & Model 2: Machine Learning (TF-IDF + Logistic Regression)

print("\n--- Evaluating Machine Learning Model (Logistic Regression) ---")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['Clean_Text'])
y = df['Sentiment']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred = lr_model.predict(X_test)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# 4. Export Extracted Data
# Save the processed data for submission
extracted_data = df[['Id', 'Clean_Text', 'Sentiment', 'Vader_Prediction']]
extracted_data.to_csv('Extracted_Data_Charanpal.csv', index=False)
print("\nExtracted data saved to 'Extracted_Data_Charanpal.csv'")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/17f50d36-54d6-4c27-a64f-
[nltk_data]     e7f6ec902d19/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/17f50d36-54d6-4c27-a64f-
[nltk_data]     e7f6ec902d19/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading and preprocessing data...

--- Evaluating Lexicon-Based Model (VADER) ---
VADER Accuracy: 0.8588
              precision    recall  f1-score   support

           0       0.60      0.35      0.44      1462
           1       0.89      0.95      0.92      7707

    accuracy                           0.86      9169
   macro avg       0.74      0.65      0.68      9169
weighted avg       0.84      0.86      0.84      9169


--- Evaluating Machine Learning Model (Logistic Regression) ---
Logistic Regression Accuracy: 0.8680
              precision    recall  f1-score   support

           0       0.93      0.22      0.35       302
           1       0.87      1.00      0.93      1532

    accuracy                           0.87      1834
   macro avg       0.90      0.61      0.64      1834
weighted avg       0.88      0.87      0.83      1834


Extracted data saved to 'Extracted_Data_Charanpal.csv'
